In [23]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn import multioutput
from sklearn import ensemble


In [24]:
data = pd.read_excel("data/clarified_dataset.xlsx")
data.head()


,Unnamed: 0,protocol_id,patient_id,suffix,measurement_number,total_measurements,last_measurements,sitting_position,unc_th_cls,th_cls,...,MG059,MG060,MG061,MG062,ocls,split_col,th_known,breast_id,year_calc,clarified
0,58,0002AA04795A,0002AA04795,A,1,1,True,0,0,0,...,-0.37,-0.07,0.8,1.5,0,0_0_2_0,True,0002AA04795A_r,34.0,0
1,66,0002AA04804A,0002AA04804,A,1,2,False,0,5,5,...,-1.07,0.03,1.1,1.4,1,5_2_2_0,True,0002AA04804A_r,57.0,0
2,67,0002AA04805A,0002AA04805,A,1,1,True,0,0,0,...,-0.61,0.59,1.0,0.8,0,0_1_3_0,True,0002AA04805A_r,36.0,0
3,68,0002AA04806A,0002AA04806,A,1,10,False,0,0,0,...,-0.29,0.31,0.5,0.6,0,0_1_2_0,True,0002AA04806A_r,35.0,0
4,72,0002AA03578B,0002AA03578,B,1,3,False,0,0,0,...,-0.59,0.11,0.5,0.6,0,0_0_2_0,True,0002AA03578B_r,32.0,0


Просто линейная регрессия

In [2]:
def testLine(df, X_train, X_test, y_train, y_test):

    reg = LinearRegression()
    reg.fit(X_train, y_train)

    y_pred = reg.predict(X_test)
    y_1 = reg.predict(X_train)
    print(f"проверка на данных обучения: {mean_absolute_error(y_train, y_1)}")
    return mean_absolute_error(y_test, y_pred)

In [3]:
df = data[data['th_cls_breast'] == 0].copy()

x_cols = [f'ir_{i}' for i in range(0, 9)]
y_cols = [f'mw_{i}' for i in range(0, 9)]
X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2, random_state=42)
print(testLine(df, X_train, X_test, y_train, y_test))

проверка на данных обучения: 0.3810936069106913
0.39184026005255496


Линейная регрессия с масштабированием

In [4]:
def testLineM(X_train, X_test, y_train, y_test):
    mms = MinMaxScaler()
    mms.fit(X_train)
    X_train_n = mms.transform(X_train)
    X_test_n = mms.transform(X_test)

    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s = scaler.transform(X_test)

    print("МИН МАКС")
    print(testLine(df, X_train_n, X_test_n, y_train, y_test))
    print("СТАНДАРТ СКЕЙЛЕР")
    print(testLine(df, X_train_s, X_test_s, y_train, y_test))

In [5]:

df = data[data['th_cls_breast'] == 0].copy()

x_cols = [f'ir_{i}' for i in range(0, 9)]
y_cols = [f'mw_{i}' for i in range(0, 9)]
X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2, random_state=42)

testLineM(X_train, X_test, y_train, y_test)

МИН МАКС
проверка на данных обучения: 0.381093606910691
0.3918402600525548
СТАНДАРТ СКЕЙЛЕР
проверка на данных обучения: 0.3810936069106911
0.39184026005255473


Добавил среднее, максимальное и минимальное

In [6]:
df = data[data['th_cls_breast'] == 0].copy()

ir_cols = [f'ir_{i}' for i in range(0, 9)]

df['ir_min'] = df[ir_cols].min(axis=1)
df['ir_max'] = df[ir_cols].max(axis=1)
df['ir_std'] = df[ir_cols].std(axis=1)

x_cols = ['ir_min','ir_max','ir_std'] + ir_cols
y_cols = [f'mw_{i}' for i in range(0, 9)]
X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2, random_state=42)

testLineM(X_train, X_test, y_train, y_test)

МИН МАКС
проверка на данных обучения: 0.3753664461887198
0.38558097667858593
СТАНДАРТ СКЕЙЛЕР
проверка на данных обучения: 0.3753664461887198
0.3855809766785858


Добавил квадраты температур

In [7]:
df = data[data['th_cls_breast'] == 0].copy()

ir_cols = [f'ir_{i}' for i in range(0, 9)]

df['ir_min'] = df[ir_cols].min(axis=1)
df['ir_max'] = df[ir_cols].max(axis=1)
df['ir_std'] = df[ir_cols].std(axis=1)

df_squared = df[ir_cols] ** 2
df_squared.columns = [f"{c}_sq" for c in ir_cols]
df = pd.concat([df, df_squared], axis=1)
squared_names = df_squared.columns.tolist()

x_cols = ['ir_min','ir_max','ir_std'] + ir_cols + squared_names
y_cols1 = [f'mw_{i}' for i in range(0, 9)]

X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2, random_state=42)

testLineM(X_train, X_test, y_train, y_test)

МИН МАКС
проверка на данных обучения: 0.35899858272288676
0.36736432882397874
СТАНДАРТ СКЕЙЛЕР
проверка на данных обучения: 0.3589985827228867
0.3673643288239784


Убрал самые крайние выбросы

In [8]:
df = data[data['th_cls_breast'] == 0].copy()

ir_cols = [f'ir_{i}' for i in range(0, 9)]

df['ir_min'] = df[ir_cols].min(axis=1)
df['ir_max'] = df[ir_cols].max(axis=1)
df['ir_std'] = df[ir_cols].std(axis=1)

df_squared = df[ir_cols] ** 2
df_squared.columns = [f"{c}_sq" for c in ir_cols]
df = pd.concat([df, df_squared], axis=1)
squared_names = df_squared.columns.tolist()

x_cols = ['ir_min','ir_max','ir_std'] + ir_cols + squared_names

y_cols1 = [f'mw_{i}' for i in range(0, 9)]

X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2, random_state=42)
percent = 0.001
lower = X_train[ir_cols].quantile(percent)
upper = X_train[ir_cols].quantile(1-percent)

train_mask = ((X_train[ir_cols] >= lower) & (X_train[ir_cols] <= upper)).all(axis=1)

X_train_filtered = X_train[train_mask].copy()
y_train_filtered = y_train[train_mask].copy()
testLineM(X_train_filtered, X_test, y_train_filtered, y_test)

МИН МАКС
проверка на данных обучения: 0.357514770244824
0.36640190226331165
СТАНДАРТ СКЕЙЛЕР
проверка на данных обучения: 0.357514770244824
0.3664019022633118


KNN

In [35]:
def testKNNM(df, X_train, X_test, y_train, y_test, n):

    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s = scaler.transform(X_test)

    knn2 = KNeighborsRegressor(n_neighbors=n)

    knn2.fit(X_train_s, y_train)

    y_1_s = knn2.predict(X_train_s)

    y_2_s = knn2.predict(X_test_s)

    print("СКЕЙЛЕР")
    print(f"На обучающей:{mean_absolute_error(y_train, y_1_s)}")
    print(f"На тестовой:{mean_absolute_error(y_test, y_2_s)}")
    return mean_absolute_error(y_test, y_2_s)

In [37]:
df = data[data['th_cls_breast'] == 0].copy()

ir_cols = [f'ir_{i}' for i in range(0, 9)]
x_cols =  ir_cols
X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2,random_state=42 )
mae = testKNNM(df, X_train, X_test, y_train, y_test, 31)


СКЕЙЛЕР
На обучающей:0.3631137944079767
На тестовой:0.3815663300797639


In [38]:
def test_knn_optimized(X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s = scaler.transform(X_test)

    best_mae = 40
    best_k = 0

    for k in range(1, 50, 2): #
        knn = KNeighborsRegressor(n_neighbors=k, weights='distance')
        knn.fit(X_train_s, y_train)
        y_pred = knn.predict(X_test_s)
        mae = mean_absolute_error(y_test, y_pred)

        if mae < best_mae:
            best_mae = mae
            best_k = k

    return best_k, best_mae

In [39]:
df = data[data['th_cls_breast'] == 0].copy()

ir_cols = [f'ir_{i}' for i in range(0, 9)]

df['ir_min'] = df[ir_cols].min(axis=1)
df['ir_max'] = df[ir_cols].max(axis=1)
df['ir_std'] = df[ir_cols].std(axis=1)

x_cols = ['ir_min','ir_max','ir_std'] + ir_cols
y_cols = [f'mw_{i}' for i in range(0, 9)]

X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2,random_state=42 )

percent = 0.001
lower = X_train[ir_cols].quantile(percent)
upper = X_train[ir_cols].quantile(1-percent)

train_mask = ((X_train[ir_cols] >= lower) & (X_train[ir_cols] <= upper)).all(axis=1)

X_train_filtered = X_train[train_mask].copy()
y_train_filtered = y_train[train_mask].copy()

result = 1
min_n = 0
best_k, mae = test_knn_optimized(X_train_filtered, X_test, y_train_filtered, y_test)
print(best_k, mae)

31 0.3794455063322414


ДЕРЕВО РЕШЕНИЙ

In [26]:

df = data[data['th_cls_breast'] == 0].copy()
ir_cols = [f'ir_{i}' for i in range(0, 9)]

df['ir_min'] = df[ir_cols].min(axis=1)
df['ir_max'] = df[ir_cols].max(axis=1)
df['ir_std'] = df[ir_cols].std(axis=1)

x_cols = ir_cols + ['ir_min','ir_max','ir_std']
y_cols = [f'mw_{i}' for i in range(0, 9)]

X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2,random_state=42 )


reg = multioutput.MultiOutputRegressor(ensemble.RandomForestRegressor())
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.3764189149985496


Градиентный бустинг

In [28]:

df = data[data['th_cls_breast'] == 0].copy()
ir_cols = [f'ir_{i}' for i in range(0, 9)]

df['ir_min'] = df[ir_cols].min(axis=1)
df['ir_max'] = df[ir_cols].max(axis=1)
df['ir_std'] = df[ir_cols].std(axis=1)

x_cols = ir_cols + ['ir_min','ir_max','ir_std']
y_cols = [f'mw_{i}' for i in range(0, 9)]

X_train, X_test, y_train, y_test = train_test_split(df[x_cols], df[y_cols], test_size=0.2,random_state=42 )

reg = multioutput.MultiOutputRegressor(ensemble.GradientBoostingRegressor())

reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.3668505298152303
